In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
os.chdir('/zhome/71/c/146676/main/')
import SimpleITK as sitk
from loaders import loader_XA_to_NA
import importlib
from helpers import module_auxiliary as ma
importlib.reload(loader_XA_to_NA)
import tifffile
from matplotlib_scalebar.scalebar import ScaleBar
from scipy.ndimage import median_filter # For extract edge function
from cil.framework import ImageGeometry, ImageData # For extract edge function
from cil.optimisation.operators import GradientOperator # For extract edge function

In [ ]:
reg = loader_XA_to_NA.load_registered_data(compression = 1, dataset_XA='tv', dataset_NA='dtv')

In [ ]:
XA = sitk.GetArrayFromImage(reg.moving)
NA = sitk.GetArrayFromImage(reg.fixed)
importlib.reload(ma)
edge = ma.extract_edge(NA, axis=1)
edge_XA = ma.extract_edge(XA,axis=1, XA=True)

In [ ]:
nz, ny, nx = np.shape(NA)
range_surf = np.arange(-40,20)
NA_surf_volume = np.empty((nz,len(range_surf), nx))
XA_surf_volume = np.empty((nz,len(range_surf), nx))
for i in range(0, len(range_surf)):
    NA_surf_volume[:,i,:] = NA[np.arange(nz)[:, None], edge+range_surf[i], np.arange(nx)]
    XA_surf_volume[:,i,:] = XA[np.arange(nz)[:, None], edge_XA+range_surf[i], np.arange(nx)]

NA_basemean = np.mean(NA_surf_volume[:,range_surf<-20])
XA_basemean = np.mean(XA_surf_volume[:,range_surf<-20])
for i in range(0,np.sum(range_surf<=0)):
    NA_surf_volume[:,i] = NA_surf_volume[:,i] - np.mean(NA_surf_volume[:,i]) + NA_basemean
    XA_surf_volume[:,i] = XA_surf_volume[:,i] - np.mean(XA_surf_volume[:,i]) + XA_basemean

In [ ]:
plt.imshow(XA[1330,:,:])

In [ ]:
plt.imshow(edge)
#plt.clim([220,320])
plt.colorbar()
plt.savefig('output.png',dpi=200)
plt.show()
plt.imshow(edge_XA)
#plt.clim([220,320])
plt.colorbar()
plt.savefig('output.png',dpi=200)
plt.show()

In [ ]:
plt.imshow(XA_surf_volume[:,36,:])
#plt.clim([220,320])
plt.colorbar()
plt.savefig('output.png',dpi=200)
plt.show()
plt.imshow(NA_surf_volume[:,36,:])
#plt.clim([220,320])
plt.colorbar()
plt.savefig('output.png',dpi=200)
plt.show()

In [ ]:
#np.save('/dtu-compute/msaca/cache/NA_surf1.npy', NA_surf_volume.astype(np.float32))
#np.save('/dtu-compute/msaca/cache/XA_surf1.npy', XA_surf_volume.astype(np.float32))

In [ ]:
NA_surf_volume = np.load('/dtu-compute/msaca/cache/NA_surf1.npy')
XA_surf_volume = np.load('/dtu-compute/msaca/cache/XA_surf1.npy')

In [ ]:
NA_surf_volume = NA_surf_volume[200:1500,30:40,200:1500]
XA_surf_volume = XA_surf_volume[200:1500,30:40,200:1500]

In [ ]:
def xi_vector_field(image_np,eta):
        nz, ny, nx = np.shape(image_np)
        ig = ImageGeometry(voxel_num_x=nx, voxel_num_y=ny, voxel_num_z=nz)
        image = ImageData(image_np.astype(np.float32), geometry=ig)
        G = GradientOperator(ig)
        numerator = G.direct(image)
        denominator = np.sqrt(eta**2 + numerator.get_item(0)**2 + numerator.get_item(1)**2 + numerator.get_item(2)**2)
        xi = numerator/denominator
        return np.sqrt(xi.get_item(0).as_array()**2 + xi.get_item(1).as_array()**2 + xi.get_item(2).as_array()**2)

NA_surf_edges = xi_vector_field(NA_surf_volume,0.005)
XA_surf_edges = xi_vector_field(XA_surf_volume,0.005)

In [ ]:
import SimpleITK as sitk

def apply_filters(volume):
    sitk_volume = sitk.GetImageFromArray(volume)

    # Bilateral Filtering
    bilateral = sitk.BilateralImageFilter()
    bilateral.SetDomainSigma(2.0)
    bilateral.SetRangeSigma(50.0)
    bilateral_filtered = sitk.GetArrayFromImage(bilateral.Execute(sitk_volume))
    print('Bil filter done')
    # Anisotropic Diffusion
    diffusion = sitk.GradientAnisotropicDiffusionImageFilter()
    diffusion.SetTimeStep(0.0625)
    diffusion.SetConductanceParameter(2.0)
    diffusion.SetNumberOfIterations(10)
    diff_filtered = sitk.GetArrayFromImage(diffusion.Execute(sitk_volume))
    print('Diffusion filter done')

    return bilateral_filtered, diff_filtered


NA_filtered = apply_filters(NA_surf_volume)
XA_filtered = apply_filters(XA_surf_volume)

In [ ]:
plt.imshow(NA_surf_volume[:,5])
plt.show()
plt.imshow(NA_filtered[1][:,5])
plt.show()
plt.imshow(NA_surf_volume[:,5] - NA_filtered[1][:,5])
plt.colorbar()
plt.show()
plt.imshow(NA_surf_edges[:,5])
plt.show()
plt.imshow(NA_filtered[0][:,5])
plt.show()


plt.imshow(XA_surf_volume[:,5])
plt.show()
plt.imshow(XA_filtered[1][:,5])
plt.show()
plt.imshow(XA_surf_volume[:,5] - XA_filtered[1][:,5])
plt.colorbar()
plt.show()
plt.imshow(XA_surf_edges[:,5])
plt.show()
plt.imshow(XA_filtered[0][:,5])
plt.show()

In [ ]:
from skimage.filters import threshold_otsu

def multi_threshold(volumes):
    thresholds = [threshold_otsu(vol) for vol in volumes]
    segmented = [(vol > th).astype(np.uint8) for vol, th in zip(volumes, thresholds)]
    return segmented

# Apply multi-thresholding on filtered versions
NA_segments = multi_threshold(NA_filtered)
XA_segments = multi_threshold(XA_filtered)

In [ ]:
XA_surf_edges_2 = sitk.VectorMagnitude(sitk.Gradient(XA_surf_volume_))

In [ ]:
NA_surf_edges_ = sitk.GetImageFromArray(NA_surf_edges)
XA_surf_edges_ = sitk.GetImageFromArray(XA_surf_edges)
XA_watershed = sitk.MorphologicalWatershed(XA_surf_edges_, level=0.25)
XA_watershed2 = sitk.MorphologicalWatershed(sitk.GetImageFromArray(XA_filtered[0]), level=0.001)
XA_watershed3 = sitk.MorphologicalWatershed(sitk.GetImageFromArray(XA_filtered[1]), level=0.001)



In [ ]:
plt.imshow(XA_surf_volume[:,5])
plt.show()

plt.imshow(NA_surf_volume[:,5])
plt.show()

plt.imshow(XA_surf_edges[:,5])
plt.show()

plt.imshow(sitk.GetArrayFromImage(XA_surf_edges_2[:,5]))
plt.colorbar()
plt.show()

plt.imshow(NA_surf_edges[:,5])
plt.show()

plt.imshow(sitk.GetArrayFromImage(XA_watershed)[:,5])
plt.colorbar()
plt.show()


plt.imshow(sitk.GetArrayFromImage(XA_watershed2)[:,5])
plt.colorbar()
plt.show()

plt.imshow(sitk.GetArrayFromImage(XA_watershed3)[:,5])
plt.colorbar()
plt.show()

In [ ]:
def apply_filters(volume,cond):
    # Anisotropic Diffusion
    diffusion = sitk.GradientAnisotropicDiffusionImageFilter()
    diffusion.SetTimeStep(0.0625)
    diffusion.SetConductanceParameter(cond*1.0)
    diffusion.SetNumberOfIterations(15)
    return diffusion.Execute(volume)


In [ ]:
level_arr = [0.1,0.12, 0.15, 0.2]


for level in level_arr:
    XA_filtered = apply_filters(XA_surf_volume_,0.5)
    XA_surf_edges = xi_vector_field(sitk.GetArrayFromImage(XA_filtered),0.005)
    XA_surf_edges_ = sitk.GetImageFromArray(XA_surf_edges)
    XA_watershed = sitk.MorphologicalWatershed(XA_surf_edges_, level=level)
    print('Finished watershed')
    fig, ax = plt.subplots(figsize=(10,10))

    # Plot the first image with a specific colormap
    ax.imshow(sitk.GetArrayFromImage(XA_filtered)[:,5], cmap='gray', alpha=1)

    # Plot the second image on top with a different colormap
    ax.imshow(sitk.GetArrayFromImage(XA_watershed)[:,5], alpha=0.2)

In [ ]:
XA_filtered = apply_filters(XA_surf_volume_,0.5)
XA_surf_edges = xi_vector_field(sitk.GetArrayFromImage(XA_filtered),0.005)
XA_surf_edges_ = sitk.GetImageFromArray(XA_surf_edges)

watershed_filter = sitk.MorphologicalWatershedImageFilter()
watershed_filter.SetMarkWatershedLine(False)  # Prevent marking edges as lines
watershed_filter.SetFullyConnected(False)  # Use fully connected components for better segmentation
watershed_filter.SetLevel(0.15)

    # Perform watershed segmentation using seeds
XA_watershed = watershed_filter.Execute(XA_surf_edges_)



In [ ]:
plt.hist(pwc.flatten(), bins=40)

In [ ]:
def place_medians_in_watersheds(image_np,eta_edge = 0.005,level = 0.15, conductivity = 0.5, smoothing_iter = 15, watershed_line = False, verbose=False):
    image = sitk.GetImageFromArray(image_np)


    diffusion = sitk.GradientAnisotropicDiffusionImageFilter()
    diffusion.SetTimeStep(0.0625)
    diffusion.SetConductanceParameter(conductivity*1.0)
    diffusion.SetNumberOfIterations(smoothing_iter)
    image_fil = diffusion.Execute(image)


    edges = xi_vector_field(sitk.GetArrayFromImage(image_fil),eta_edge)
    edges_ = sitk.GetImageFromArray(edges)
    watershed_filter = sitk.MorphologicalWatershedImageFilter()
    watershed_filter.SetMarkWatershedLine(watershed_line)  # Prevent marking edges as lines
    watershed_filter.SetFullyConnected(False)  # Use fully connected components for better segmentation
    watershed_filter.SetLevel(level)

    # Perform watershed segmentation using seeds
    watershed_classes_ = watershed_filter.Execute(edges_)
    watershed_classes = sitk.GetArrayFromImage(watershed_classes_)
    pwc = np.zeros(np.shape(watershed_classes))
    for i in range(1,np.max(watershed_classes)):
        pwc[watershed_classes==i] = np.median(image_np[watershed_classes==i])

    if verbose:
        print('Input -> Diffusion -> Edge calculation -> Watershed -> Assigning medians to watersheds')
        print('The number of watersheds is ',np.max(watershed_classes))

        print('Input image:')
        plt.imshow(image_np[:,5])
        plt.show()
        print('Diffused image')
        plt.imshow(sitk.GetArrayFromImage(image_fil)[:,5])
        plt.show()
        print('Extracted edges image')
        plt.imshow(edges[:,5])
        plt.show()
        print('Watersheds including watershed seperating lines')
        watershed_filter.SetMarkWatershedLine(True)
        watershed_filter.SetFullyConnected(False)  # Use fully connected components for better segmentation
        watershed_filter.SetLevel(level)
        # Perform watershed segmentation using seeds
        watershed_classes_ = watershed_filter.Execute(edges_)
        watershed_classes = sitk.GetArrayFromImage(watershed_classes_)
        plt.imshow(watershed_classes[:,5])
        plt.colorbar()
        plt.show()
    return pwc


In [ ]:
XA_pwc = place_medians_in_watersheds(XA_surf_volume)
plt.imshow(XA_pwc[:,5])
plt.show()

In [ ]:
NA_pwc = place_medians_in_watersheds(NA_surf_volume,level=0.1)
plt.imshow(NA_pwc[:,5])
plt.show()

In [ ]:
import matplotlib.cm as cm
import matplotlib.colors as mcolors

def add_squares(x, y):
  
    """
    Add squares to an existing figure based on specified intervals, with automatic color assignment.
    
    Parameters:
    - x: List of intervals for the x-axis, e.g., [[x1_s, x1_e], [x2_s, x2_e], ...].
    - y: List of intervals for the y-axis, e.g., [[y1_s, y1_e], [y2_s, y2_e], ...].
    - cmap: Matplotlib colormap for generating colors (default: "tab10").
    - linewidth: Thickness of the square outlines (default: 2).

    Returns:
    - colors: List of colors assigned to each square.
    """
    cmap = "tab10"
    if len(x) != len(y):
        raise ValueError("The number of x-intervals and y-intervals must be the same.")
    
    ax = plt.gca()  # Get the current axes
    num_squares = len(x)
    colors = cm.get_cmap(cmap, num_squares).colors  # Generate distinct colors from the colormap
    
    for i, (x_interval, y_interval) in enumerate(zip(x, y)):
        # Ensure intervals are valid
        if len(x_interval) != 2 or len(y_interval) != 2:
            raise ValueError("Each interval must have exactly two elements: [start, end].")
        
        # Add a rectangle for each interval pair
        rectangle = plt.Rectangle(
            (x_interval[0], y_interval[0]),  # Bottom-left corner
            x_interval[1] - x_interval[0],  # Width
            y_interval[1] - y_interval[0],  # Height
            edgecolor=colors[i],
            facecolor="none",
            linewidth=2,
        )
        ax.add_patch(rectangle)

    colormaps = []
    for i, color in enumerate(colors):
        # Define the color map
        name_prefix = "custom"
        factor = 1.5
        cmap = mcolors.LinearSegmentedColormap.from_list(
            f"{name_prefix}_{i}",
            [(0, 0, 0), color]  # Transition from black to the given color
        )
        cdict = cmap._segmentdata
        # Modify the colors (this is done for each color channel: red, green, blue)
        for channel in ['red', 'green', 'blue']:
            # Scale each color in the channel towards 1 (white)
            for i, (t, c1, c2) in enumerate(cdict[channel]):
                new_c1 = min(c1 * factor, 1.0)
                new_c2 = min(c2 * factor, 1.0)
                cdict[channel][i] = (t, new_c1, new_c2)
        b_cmap = mcolors.LinearSegmentedColormap(cmap.name + '_brightened', segmentdata=cdict)
        colormaps.append(b_cmap)
    return colormaps


def heatmap(fixed_values, moving_values,neutron=None, xray = None):
    bins = 100
    nticks = 15
    range_fixed = [-0.01, 0.11]
    range_moving = [-0.01, 0.11]

    # Create the heatmap
    heatmap, xedges, yedges = np.histogram2d(
        fixed_values, moving_values, bins=(bins, bins),
        range=[range_fixed, range_moving]
    )
    log_heatmap = np.log1p(heatmap)

    # Plot the heatmap
    plt.figure(figsize=(10, 10))
    plt.imshow(
        log_heatmap.T, origin="lower", aspect="auto", cmap="YlGnBu",
        extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]]  # Set extent to match data range
    )
    plt.colorbar(label="log-Frequency")
    plt.xlabel("Fixed: Neutron")
    plt.ylabel("Moving: Xray")
    plt.title("Heatmap with Highlighted Square")

    # Set custom ticks
    x_tick_positions = np.linspace(xedges[0], xedges[-1], num=nticks)
    y_tick_positions = np.linspace(yedges[0], yedges[-1], num=nticks)
    x_tick_labels = [f"{x:.3f}" for x in x_tick_positions]
    y_tick_labels = [f"{y:.3f}" for y in y_tick_positions]
    plt.xticks(x_tick_positions, x_tick_labels)
    plt.yticks(y_tick_positions, y_tick_labels)

    if neutron is not None:
        colors = add_squares(x=neutron, y=xray)
    # Show the plot
    plt.show()
    if neutron is not None:
        return colors

In [ ]:
neutron = [[-0.01, 0.05], [-0.01,0.02], [0.03, 0.1], [-0.01,0.02], [0.03, 0.1]]
xray = [[-0.01,0.0], [0, 0.025], [0,0.025], [0.035, 0.1], [0.035, 0.1]]
colors = heatmap(NA_pwc.flatten(), XA_pwc.flatten(),neutron=neutron, xray = xray)

In [ ]:
nz, ny, nx = np.shape(XA_pwc)
segm = np.empty((len(neutron), nz, ny, nx), dtype=bool)
for i in range(len(neutron)):
    segm[i] = np.logical_and(NA_pwc>neutron[i][0], NA_pwc<neutron[i][1])*np.logical_and(XA_pwc>xray[i][0],XA_pwc<xray[i][1])

segm_classes = np.empty((nz, ny, nx))
for i in range(len(neutron)):
    segm_classes[segm[i]] = i+1

In [ ]:
import matplotlib.colors as mcolors
cmap = mcolors.ListedColormap(['none', 'yellow', 'green', 'blue', 'red', 'purple'])  # Class 0 is 'none' (transparent)
bounds = [0, 1, 2, 3, 4, 5]  # Define boundaries for the colormap
norm = mcolors.BoundaryNorm(bounds, cmap.N)

fig, ax = plt.subplots(figsize=(10,10))

# Plot the first image with a specific colormap
ax.imshow(XA_surf_volume[:,5], cmap='gray', interpolation='none')


ax.imshow(segm_classes[:,5], cmap=cmap, norm=norm, interpolation='none', alpha=(segm_classes[:,8] > 0).astype(float)*0.2)

# Hide axes
ax.set_xticks([])
ax.set_yticks([])

# Show plot
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(NA_surf, cmap='gray', vmin=-0.01, vmax=0.07)

scalebar = ScaleBar(6.12, "µm", location="lower right", color="white", scale_loc="bottom", box_alpha=0.5)
ax.add_artist(scalebar)

plt.savefig('/dtu-compute/msaca/sliceA_eds/EDS_BB_A_raw_files/NA_TV_surface.png',dpi = 250 )

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(XA_surf, cmap='gray', vmin=-0.01, vmax=0.07)

scalebar = ScaleBar(6.12, "µm", location="lower right", color="white", scale_loc="bottom", box_alpha=0.5)
ax.add_artist(scalebar)
plt.savefig('/dtu-compute/msaca/sliceA_eds/EDS_BB_A_raw_files/XA_surface.png',dpi = 250 )

In [ ]:
xray_slices = range(0,10)
neutron_slices = range(0,nz)
output_volume = [[0,nz],[None], [None]]
reg_ = loader_XA_to_NA.load_subset_of_registered_data(dataset_XA='tv', dataset_NA='dtv' , h=1, xray_slices = xray_slices,
    neutron_slices = neutron_slices, output_volume = output_volume)

In [ ]:
NA_dtv = sitk.GetArrayFromImage(reg_.fixed)
NA_dtv_surf = NA_dtv[np.arange(nz)[:, None], f_argmin_-3, np.arange(nx)]
fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(NA_dtv_surf, cmap='gray', vmin=-0.01, vmax=0.07)

scalebar = ScaleBar(6.12, "µm", location="lower right", color="white", scale_loc="bottom", box_alpha=0.5)
ax.add_artist(scalebar)

plt.savefig('/dtu-compute/msaca/sliceA_eds/EDS_BB_A_raw_files/NA_dTV_surface.png',dpi = 250 )


In [ ]:
path = '/dtu-compute/msaca/sliceA_eds/EDS_BB_A_raw_files/edsMg.tiff'
eds = tifffile.imread(path)
eds[eds>150] = 0
eds = np.fliplr(eds)
#eds = np.flipud(eds)
eds = eds[4000::2,::2]/100


In [ ]:
_ , nx_XA = np.shape(XA_surf)
_ , nx_eds = np.shape(eds)

fixed = sitk.GetImageFromArray(XA_surf)
moving = sitk.GetImageFromArray(eds)
moving_NA = sitk.GetImageFromArray(XA_surf)
fixed.SetSpacing((1/nx_XA,1/nx_XA))
moving.SetSpacing((1/nx_eds,1/nx_eds))

size_fixed = fixed.GetSize()
size_moving = moving.GetSize()

spacing_fixed = fixed.GetSpacing()
spacing_moving = moving.GetSpacing()

# Compute the new origin (shift it to -N/2)
new_origin_fixed = [-0.5 * (size_fixed[i] - 1) * spacing_fixed[i] for i in range(len(size_fixed))]
new_origin_moving = [-0.5 * (size_moving[i] - 1) * spacing_moving[i] for i in range(len(size_moving))]


fixed.SetOrigin(new_origin_fixed)
moving.SetOrigin(new_origin_moving)

In [ ]:
factor = 3

transform = sitk.Transform(2, sitk.sitkIdentity)
# Get the original size and spacing of the image
size = moving.GetSize()
spacing = moving.GetSpacing()

# Calculate the new size (downsampling by factor)
new_size = [int(size[0] / factor), int(size[1] / factor)]
# Calculate the new spacing (enlarging the spacing to match the downsampled size)
new_spacing = [s * factor for s in spacing]

# Perform the resampling (using average interpolation for downsampling)
moving_d = sitk.Resample(moving,
                                new_size,
                                transform,
                                sitk.sitkLinear,  # BSpline interpolation is good for downsampling
                                moving.GetOrigin(),
                                new_spacing,
                                moving.GetDirection(),
                                0)  # 0 is the background value for the resampling


In [ ]:
def registrator(fixed, moving, thresholds = [0.025, 30], sampling_percentage = 0.1, max_iter = 1000, learning_rate = 1):

    fixed_d = sitk.BinaryThreshold(fixed, lowerThreshold=thresholds[0], upperThreshold=float("inf"), insideValue=1, outsideValue=0)
    moving_d = sitk.BinaryThreshold(moving, lowerThreshold=thresholds[1], upperThreshold=float("inf"), insideValue=1, outsideValue=0)
    fixed_d =sitk.Cast(fixed_d, sitk.sitkFloat32)
    moving_d =sitk.Cast(moving_d, sitk.sitkFloat32)

    initial_transform = sitk.Similarity2DTransform()
    initial_transform.SetMatrix([1.0, 0.0,
                                0.0, 1.0])

    # Set the translation to zero
    initial_transform.SetTranslation([0.0, 0.0])
    initial_transform.SetScale(1.0)

    registration = sitk.ImageRegistrationMethod()
    registration.SetInitialTransform(initial_transform)
    registration.SetMetricAsMeanSquares()
    registration.SetMetricSamplingStrategy(registration.RANDOM)
    registration.SetMetricSamplingPercentage(sampling_percentage)

    registration.SetOptimizerAsGradientDescent(
        learningRate=learning_rate,
        numberOfIterations=max_iter,
        convergenceMinimumValue=-1e-16,
        convergenceWindowSize=1000
        )

    registration.SetInterpolator(sitk.sitkLinear)
    registration.Execute(fixed_d, moving_d)
    transform = registration.GetInitialTransform()
    return transform

def resampler(fixed, moving, transform):
    resampler = sitk.ResampleImageFilter()
    resampler.SetReferenceImage(fixed)  # Reference image (fixed)
    resampler.SetInterpolator(sitk.sitkLinear)   # Interpolation method
    resampler.SetTransform(transform)    # Apply the initial transform (aligned centroids)
    resampler.SetOutputPixelType(fixed.GetPixelID())
    resampler.SetOutputSpacing(fixed.GetSpacing())  # Ensure the spacing is preserved
    resampler.SetOutputOrigin(fixed.GetOrigin())  # Preserve origin
    resampler.SetOutputDirection(fixed.GetDirection())
    moving = resampler.Execute(moving)
    return moving

In [ ]:
transforms = []

transforms.append(registrator(fixed, moving_d, thresholds = [0.025, 0.20], sampling_percentage = 0.2, max_iter = 500, learning_rate = 3))
moving_temp = resampler(fixed, moving_d, transforms[0])



transforms.append(registrator(fixed, moving_temp, thresholds = [0.025, 0.20], sampling_percentage = 0.2, max_iter = 500, learning_rate = 1))
moving_temp = resampler(fixed, moving_temp, transforms[1])



transforms.append(registrator(fixed, moving_temp, thresholds = [0.025, 0.20], sampling_percentage = 0.2, max_iter = 500, learning_rate = 0.5))
moving_temp = resampler(fixed, moving_temp, transforms[2])


transforms.append(registrator(fixed, moving_temp, thresholds = [0.025, 0.20], sampling_percentage = 0.2, max_iter = 500, learning_rate = 0.2))
moving_temp = resampler(fixed, moving_temp, transforms[3])

transforms.append(registrator(fixed, moving_temp, thresholds = [0.025, 0.20], sampling_percentage = 0.2, max_iter = 500, learning_rate = 0.1))
moving_temp = resampler(fixed, moving_temp, transforms[4])



In [ ]:
m1 = sitk.GetArrayFromImage(moving_temp)>0.2
m2 = sitk.GetArrayFromImage(fixed)>0.025
diff_ = m1*1-m2*1
plt.imshow(diff_)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(sitk.GetArrayFromImage(moving_temp), cmap='gray', vmin=0, vmax=1)

scalebar = ScaleBar(6.12, "µm", location="lower right", color="white", scale_loc="bottom", box_alpha=0.5)
ax.add_artist(scalebar)

plt.savefig('/dtu-compute/msaca/sliceA_eds/EDS_BB_A_raw_files/N_eds_registered_to_XA_NA.png',dpi = 250 )

